# Whisper-small + LoRA on PriMock57 (free Colab or Kaggle T4)

Fine-tunes `openai/whisper-small` with LoRA on the **train** consultations of PriMock57
(CC BY 4.0) and reports WER/CER on the **held-out test consultations**, before and after,
with consultation-level bootstrap CIs and a paired comparison.

* **Colab:** Runtime → Change runtime type → *T4 GPU*.
* **Kaggle:** Settings → *Accelerator: GPU T4*, *Internet: on*.

Then Run all. Each step is a `%%bash` cell with `set -e`, so if a step fails, the run stops
there instead of continuing on bad data. Roughly 1-1.5 h end to end on a T4.

In [ ]:
%%bash
set -e
nvidia-smi --query-gpu=name,memory.total --format=csv
[ -d medscribe ] || [ -f pyproject.toml ] || git clone https://github.com/vik-bit/medscribe.git

In [ ]:
import os

if not os.path.exists("pyproject.toml"):  # safe to re-run
    os.chdir("medscribe")
print(os.getcwd())

In [ ]:
%%bash
set -e
pip install -q -e ".[asr]"
# ~1.7 GB of audio at a pinned commit; uses git-lfs if present, otherwise downloads each
# file directly and verifies its sha256. Fails if any file is still a git-lfs pointer.
bash scripts/fetch_primock57.sh
python -m medscribe splits

## 1. Baseline (zero-shot whisper-small) on the full test split

In [ ]:
%%bash
set -e
python -m medscribe asr-eval --split test --batch-size 16     --out results/asr_test_base.json --preds results/asr_test_base.jsonl

## 2. LoRA fine-tuning (plain PyTorch loop, fp16 autocast)
The best adapter is selected on the **dev** split, never on test.

In [ ]:
%%bash
set -e
python -m medscribe asr-train --epochs 3 --batch-size 16 --lr 1e-3     --lora-r 32 --lora-alpha 64 --eval-every 200 --out checkpoints/whisper-small-lora

## 3. Fine-tuned model on the same test utterances + paired comparison

In [ ]:
%%bash
set -e
python -m medscribe asr-eval --split test --batch-size 16     --adapter checkpoints/whisper-small-lora/best     --out results/asr_test_lora.json --preds results/asr_test_lora.jsonl
python -m medscribe asr-compare --a results/asr_test_lora.jsonl --b results/asr_test_base.jsonl     --out results/asr_test_compare.json
python scripts/asr_error_analysis.py results/asr_test_base.jsonl --out results/asr_test_base_errors.json > /dev/null
python scripts/asr_error_analysis.py results/asr_test_lora.jsonl --out results/asr_test_lora_errors.json > /dev/null

## 4. Download the results
Commit the `results/` files to the repo; the README reports only numbers from these files.

In [ ]:
%%bash
set -e
cp checkpoints/whisper-small-lora/train_log.json results/asr_train_log.json
zip -qr ../medscribe_asr_results.zip results/asr_test_* results/asr_train_log.json     checkpoints/whisper-small-lora/best
ls -lh ../medscribe_asr_results.zip

In [ ]:
# Colab: trigger a browser download. Kaggle: the zip is in /kaggle/working (Output tab).
try:
    from google.colab import files

    files.download("../medscribe_asr_results.zip")
except ImportError:
    print("Not on Colab; download medscribe_asr_results.zip from the output panel.")